In [ ]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../config")

import pandas as pd
import matplotlib.pyplot as plt
from config_loader import cfg

from src.data.labels import (
    label_pagan_sossounov,
    label_peak_to_trough,
    label_lunde_timmermann,
    load_nber_recession,
    compute_concordance_matrix,
    plot_label_timeline,
)
from src.data.labels.concordance import plot_concordance_heatmap

df = pd.read_parquet(cfg.data_path("test_data"))
prices = df["Cumulative_Returns"]

In [ ]:
labels = {
    "MSM":     df["MSM_Signal"].astype("int8"),
    "HMM":     df["HMM_Signal"].astype("int8"),
    "PagSoss": label_pagan_sossounov(prices),
    "P2T":     label_peak_to_trough(prices, threshold=0.20),
    "LundeT":  label_lunde_timmermann(prices),
    "NBER":    load_nber_recession(df.index),
}

for name, s in labels.items():
    print(f"{name:10s}: {s.sum()} Bear-Tage ({s.mean()*100:.1f}%), "
          f"NaN={s.isna().sum()}")

In [ ]:
concordance = compute_concordance_matrix(labels)
print(concordance.round(3))
plot_concordance_heatmap(concordance, cfg.asset_path("label_concordance_matrix"))

In [ ]:
raw = pd.read_parquet(cfg.data_path("raw"))
plot_prices = raw["^GSPC"].reindex(df.index).ffill()
plot_label_timeline(labels, plot_prices, cfg.asset_path("label_timeline_comparison"))

In [ ]:
switch_stats = pd.DataFrame({
    name: {
        "bear_share_%": s.mean() * 100,
        "n_switches": (s.diff().abs() == 1).sum(),
        "avg_phase_days": len(s) / max((s.diff().abs() == 1).sum(), 1),
    } for name, s in labels.items()
}).T
print(switch_stats.round(1))